# Example analysis

Notebooks use the same `storage` module the pipeline and the app use:
`load()` to read a table, `save()` / `save_figure()` to write one back.
Anything you save here is immediately readable by a page in `04_pages/`.

Run every notebook in this folder with `make notebooks`.

In [ ]:
import sys

sys.path.insert(0, "..")  # the notebook runs from 03_notebooks/

import matplotlib.pyplot as plt
import pandas as pd

from backend import storage

storage.tables()

In [ ]:
records = storage.load("records")
records.head()

Aggregate something, then save it under a name a page can read.

No column name is baked in below: it groups by the first label column and
sums the first numeric one, so it runs against whatever `transform.py`
produced. Replace it with your real analysis — this is just a placeholder
that always has something to draw.

In [ ]:
totals = pd.DataFrame()

if records.empty:
    print("No `records` table yet — run `make pipeline` first.")
else:
    numbers = list(records.select_dtypes("number").columns)
    labels = [column for column in records.columns if column not in numbers]

    dimension = labels[0] if labels else records.columns[0]
    measures = [column for column in numbers if column != dimension]

    totals = (
        records.groupby(dimension, as_index=False)[measures[0]].sum()
        if measures
        else records.groupby(dimension, as_index=False).size()
    ).sort_values(dimension, ignore_index=True)

    storage.save("trend_totals", totals)

totals.head()

In [ ]:
if not totals.empty:
    dimension, measure = totals.columns[0], totals.columns[1]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(range(len(totals)), totals[measure], marker="o" if len(totals) <= 40 else "")
    ax.set_title(f"{measure} by {dimension}")
    ax.set_ylabel(measure)
    ax.grid(alpha=0.3)

    # Thin the tick labels so a long axis stays readable.
    step = max(1, len(totals) // 12)
    ax.set_xticks(range(0, len(totals), step))
    ax.set_xticklabels(totals[dimension].astype(str)[::step], rotation=45, ha="right")
    fig.tight_layout()

    storage.save_figure("trend_totals", fig)